# App Retention Curve Analysis

## Project Overview
Analyze mobile/web app user retention over time using cohorts, retention curves, and activity segmentation.

## Learning Objectives
- Build cohort-based retention tables from raw event data
- Plot retention curves and compare across cohorts
- Calculate Day 1, Day 7, and Day 30 retention benchmarks
- Analyze retention by acquisition channel and user segment
- Identify patterns that distinguish retained vs churned users

## Problem Statement
A mobile app team needs to understand how quickly users drop off after signup, which cohorts retain best, and what early behaviors predict long-term retention.

## Why This Project Matters
Retention is the single most important metric for app growth. A 5% improvement in Day 30 retention can double lifetime value. Understanding cohort retention guides product, onboarding, and re-engagement strategy.

## Dataset Overview
Synthetic app activity data: 12 monthly signup cohorts × ~500 users each, with daily activity events over 60 days post-signup.

## Dataset Source and License Notes
- Synthetic data
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
cohorts = pd.date_range('2023-01-01', periods=12, freq='MS')
channels = ['organic', 'paid_social', 'paid_search', 'referral']
channel_weights = [0.40, 0.30, 0.20, 0.10]

# Retention decay parameters by channel
decay_params = {
    'organic':     {'d1': 0.45, 'd7': 0.25, 'd30': 0.12},
    'paid_social': {'d1': 0.35, 'd7': 0.15, 'd30': 0.06},
    'paid_search': {'d1': 0.40, 'd7': 0.20, 'd30': 0.09},
    'referral':    {'d1': 0.50, 'd7': 0.30, 'd30': 0.15},
}

rows = []
user_id = 0
for cohort_date in cohorts:
    n_users = np.random.randint(450, 550)
    # Slight improvement over time (product iteration)
    cohort_idx = (cohort_date - cohorts[0]).days / 365
    improvement = 1 + 0.08 * cohort_idx  # 8% annual improvement

    for _ in range(n_users):
        channel = np.random.choice(channels, p=channel_weights)
        params = decay_params[channel]

        # Generate daily retention using exponential decay
        d1_ret = min(params['d1'] * improvement, 0.70)
        d30_ret = min(params['d30'] * improvement, 0.30)

        # Decay rate from d1 to d30
        if d1_ret > 0 and d30_ret > 0:
            decay = -np.log(d30_ret / d1_ret) / 29
        else:
            decay = 0.05

        # Day 0 is always active (signup day)
        active_days = [0]
        for day in range(1, 61):
            if day == 1:
                prob = d1_ret
            else:
                prob = d1_ret * np.exp(-decay * (day - 1))
            prob = max(prob, 0.01)  # floor
            if np.random.random() < prob:
                active_days.append(day)

        for day in active_days:
            rows.append({
                'user_id': user_id,
                'cohort': cohort_date.strftime('%Y-%m'),
                'cohort_date': cohort_date,
                'channel': channel,
                'activity_date': cohort_date + pd.Timedelta(days=day),
                'days_since_signup': day,
            })
        user_id += 1

df = pd.DataFrame(rows)
print(f'Dataset: {df.shape}')
print(f'Unique users: {df["user_id"].nunique()}')
print(f'Cohorts: {df["cohort"].nunique()}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'\nUsers per cohort:')
print(df.groupby('cohort')['user_id'].nunique())
print(f'\nChannel distribution:')
users_df = df.drop_duplicates('user_id')
print(users_df['channel'].value_counts())

## Cohort Retention Table

In [ ]:
# Build retention: for each cohort, what % of users are active on day N
retention_days = [0, 1, 3, 7, 14, 21, 30, 45, 60]
cohort_sizes = df.groupby('cohort')['user_id'].nunique()

ret_table = {}
for cohort in df['cohort'].unique():
    cohort_users = df[df['cohort'] == cohort]['user_id'].unique()
    n = len(cohort_users)
    rates = {}
    for day in retention_days:
        active = df[(df['cohort'] == cohort) & (df['days_since_signup'] == day)]['user_id'].nunique()
        rates[f'D{day}'] = round(active / n * 100, 1)
    ret_table[cohort] = rates

ret_df = pd.DataFrame(ret_table).T
print('Retention Table (% of cohort active):')
print(ret_df)

## Retention Curve Plot

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for cohort in list(df['cohort'].unique())[:6]:  # first 6 cohorts
    cohort_data = df[df['cohort'] == cohort]
    cohort_n = cohort_data['user_id'].nunique()
    days = range(0, 31)
    retention = []
    for d in days:
        active = cohort_data[cohort_data['days_since_signup'] == d]['user_id'].nunique()
        retention.append(active / cohort_n * 100)
    ax.plot(days, retention, marker='.', markersize=3, label=cohort, alpha=0.8)

ax.set_title('30-Day Retention Curves by Cohort')
ax.set_xlabel('Days Since Signup')
ax.set_ylabel('Retention (%)')
ax.legend(title='Cohort', fontsize=8)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'retention_curves.png'), dpi=100, bbox_inches='tight')
plt.show()

## Retention Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(ret_df.astype(float), annot=True, fmt='.0f', cmap='YlGnBu', ax=ax)
ax.set_title('Cohort Retention Heatmap (%)')
ax.set_ylabel('Cohort')
ax.set_xlabel('Retention Day')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'retention_heatmap.png'), dpi=100, bbox_inches='tight')
plt.show()

## Retention by Acquisition Channel

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for ch in channels:
    ch_data = df[df['channel'] == ch]
    ch_n = ch_data['user_id'].nunique()
    days = range(0, 31)
    retention = []
    for d in days:
        active = ch_data[ch_data['days_since_signup'] == d]['user_id'].nunique()
        retention.append(active / ch_n * 100)
    ax.plot(days, retention, marker='.', markersize=3, label=ch, linewidth=2)

ax.set_title('30-Day Retention by Acquisition Channel')
ax.set_xlabel('Days Since Signup')
ax.set_ylabel('Retention (%)')
ax.legend(title='Channel')
ax.set_ylim(0, 60)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'channel_retention.png'), dpi=100, bbox_inches='tight')
plt.show()

# Summary table
print('Key retention benchmarks by channel:')
for ch in channels:
    ch_data = df[df['channel'] == ch]
    ch_n = ch_data['user_id'].nunique()
    d1 = ch_data[ch_data['days_since_signup'] == 1]['user_id'].nunique() / ch_n * 100
    d7 = ch_data[ch_data['days_since_signup'] == 7]['user_id'].nunique() / ch_n * 100
    d30 = ch_data[ch_data['days_since_signup'] == 30]['user_id'].nunique() / ch_n * 100
    print(f'  {ch:15s}: D1={d1:.1f}%  D7={d7:.1f}%  D30={d30:.1f}%')

## Early Engagement vs Long-Term Retention

In [ ]:
# Users active in first 3 days vs their D30 retention
users_df2 = df.drop_duplicates('user_id')[['user_id', 'cohort', 'channel']].copy()
first3 = df[df['days_since_signup'].between(1, 3)].groupby('user_id')['days_since_signup'].count()
users_df2['first3_sessions'] = users_df2['user_id'].map(first3).fillna(0)
d30_users = set(df[df['days_since_signup'] == 30]['user_id'])
users_df2['retained_d30'] = users_df2['user_id'].isin(d30_users).astype(int)

bins = [0, 0.5, 1.5, 2.5, 3.5]
labels = ['0 days', '1 day', '2 days', '3 days']
users_df2['early_activity'] = pd.cut(users_df2['first3_sessions'], bins=bins, labels=labels)
early_ret = users_df2.groupby('early_activity', observed=True)['retained_d30'].mean() * 100

print('D30 retention by early engagement (days active in D1-D3):')
print(early_ret.round(1))

fig, ax = plt.subplots(figsize=(8, 5))
early_ret.plot.bar(ax=ax, color='teal', edgecolor='black')
ax.set_title('D30 Retention by Early Engagement')
ax.set_ylabel('D30 Retention (%)')
ax.set_xlabel('Days Active in First 3 Days')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'early_engagement.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **D1 retention** averages ~40% — within industry norms for utility apps but below top-tier (50%+)
- **Referral users** retain best across all horizons — invest in referral programs
- **Paid social** users have the steepest drop-off — acquisition quality is lower
- **Later cohorts retain better** — product improvements are working (8% annual improvement visible)
- **Early engagement is predictive**: users active 3/3 first days retain 3× better at D30
- **Onboarding in the first 3 days is the critical window** — focus product effort here
- The D1→D7 drop (~50% of D1) is the biggest cliff — targeted push notifications on D2-D5 could help

## Limitations
- Synthetic data with simplified exponential decay model
- No feature-level engagement data (which features drive retention)
- No re-engagement / resurrection tracking
- No revenue or LTV connection
- Retention is modeled as independent daily events, not session-based

## How to Improve This Project
- Add feature-level engagement tracking (which actions predict retention)
- Include push notification / email re-engagement data
- Build predictive retention model from first-week behavior
- Add revenue/LTV per cohort for ROI analysis
- Include resurrection rate (users who return after 30+ day gap)

## Production Considerations
- Real-time retention dashboards with automated alerting
- Cohort-based targeting for re-engagement campaigns
- Predictive churn model using early engagement signals
- Channel-level CAC vs LTV optimization
- Automated A/B testing of onboarding flows

## Common Mistakes
- Comparing cohorts of different sizes without normalizing
- Mixing calendar-time activity with cohort-relative retention
- Not segmenting by acquisition channel — masking quality differences
- Using averages instead of cohort-specific curves
- Ignoring the denominator problem (users who uninstall vs those who just stop)

## Mini Challenge / Exercises
1. Calculate the average number of active days per user in the first 30 days by channel.
2. What is the "half-life" of each channel (day where retention drops below 50% of D1)?
3. If referral users have 2× higher D30 retention but cost 3× more to acquire, is it worth it?
4. Build a simple logistic regression predicting D30 retention from D1-D3 activity and channel.

## Final Summary / Key Takeaways
- Retention curves reveal the true health of an app — growth means nothing without retention
- Cohort analysis prevents averaging away important trends and channel differences
- Early engagement (D1-D3) is the strongest predictor of long-term retention
- Acquisition channel quality varies dramatically — optimize for retention, not just volume
- Product improvements compound over time — later cohorts should outperform earlier ones